# 보이스피싱 CSV 통합 파이프라인

업로드한 CSV를 관측 단위에 따라 두 데이터프레임으로 정리합니다.

1. `df_postal`: 우체국금융개발원 자료 중 전화 접근·개인 피해이며 일반대출이 아닌 사례
2. `df_police`: 경찰청 연도별 집계 지표를 통합한 long-format 자료

동일한 연령별 파일처럼 내용이 완전히 같은 중복 파일은 자동 제외합니다. 우체국 자료는 사건 식별자가 없으므로 동일 행을 임의 삭제하지 않고 중복 후보 플래그만 만듭니다.

In [ ]:
from pathlib import Path
import hashlib, io, re, zipfile
import numpy as np
import pandas as pd

try:
    from google.colab import files
    uploaded = files.upload()  # 분석할 CSV를 한꺼번에 선택
    csv_blobs = {name: bytes(blob) for name, blob in uploaded.items() if name.lower().endswith('.csv')}
except ImportError:
    # 로컬 실행 시 SOURCE_DIR을 원하는 폴더로 바꾸세요.
    SOURCE_DIR = Path(r'C:/Users/mbc/Downloads')
    csv_blobs = {p.name: p.read_bytes() for p in SOURCE_DIR.glob('*.csv')}

assert csv_blobs, 'CSV 파일이 없습니다.'
print(f'불러온 CSV: {len(csv_blobs)}개')
print(*sorted(csv_blobs), sep='\n- ')

In [ ]:
def read_korean_csv(blob):
    for enc in ('utf-8-sig', 'cp949', 'euc-kr'):
        try:
            df = pd.read_csv(io.BytesIO(blob), encoding=enc, dtype=str)
            df.columns = df.columns.str.strip()
            return df.apply(lambda s: s.str.strip() if s.dtype == 'object' else s), enc
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError('unknown', b'', 0, 1, '지원하지 않는 인코딩')

# SHA-256이 같은 파일은 첫 파일만 사용
seen_hashes, tables, manifest = {}, {}, []
for name, blob in sorted(csv_blobs.items()):
    digest = hashlib.sha256(blob).hexdigest()
    duplicate_of = seen_hashes.get(digest)
    df, enc = read_korean_csv(blob)
    manifest.append({'file': name, 'rows': len(df), 'columns': len(df.columns),
                     'encoding': enc, 'sha256': digest, 'duplicate_of': duplicate_of})
    if duplicate_of is None:
        seen_hashes[digest] = name
        tables[name] = df

df_manifest = pd.DataFrame(manifest)
display(df_manifest)

## 1. 우체국금융개발원 사건 단위 데이터 통합

In [ ]:
postal_required = {'사기유형', '사칭기관'}
postal_tables = []
for name, df in tables.items():
    if postal_required.issubset(df.columns) and ('연령대' in df.columns or '나이대' in df.columns):
        x = df.rename(columns={
            '나이대': '연령대', '피해자성별': '피해자 성별', '피해금액': '피해액',
            '피해자계좌_피해(송금)액': '피해액', '피해년월': '접수년월'
        }).copy()
        snapshot = re.search(r'(20\d{6})', name)
        x['자료기준일'] = snapshot.group(1) if snapshot else pd.NA
        x['원본파일'] = name
        postal_tables.append(x)

assert postal_tables, '우체국금융개발원 형식의 CSV를 찾지 못했습니다.'
df_postal_raw = pd.concat(postal_tables, ignore_index=True, sort=False)
required_filter_cols = {'접근매체', '피해자 성별', '피해구제 신청사유'}
assert required_filter_cols.issubset(df_postal_raw.columns), '우체국 필터에 필요한 컬럼이 없습니다.'
phone_mask = df_postal_raw['접근매체'].eq('전화')
person_mask = df_postal_raw['피해자 성별'].ne('기업')
non_loan_mask = df_postal_raw['피해구제 신청사유'].ne('일반대출')
filter_summary = pd.DataFrame({
    '단계': ['원본 통합', '전화 접근만', '기업 제외', '일반대출 제외(최종)'],
    '행수': [len(df_postal_raw), phone_mask.sum(), (phone_mask & person_mask).sum(),
             (phone_mask & person_mask & non_loan_mask).sum()]
})
df_postal = df_postal_raw.loc[phone_mask & person_mask & non_loan_mask].copy().reset_index(drop=True)

if '접수년월' in df_postal.columns:
    ym = df_postal['접수년월'].astype('string').str.extract(r'(20\d{2})\D*(\d{1,2})')
    df_postal['최초 접수년'] = df_postal.get('최초 접수년', ym[0]).fillna(ym[0])
    df_postal['최초 접수월'] = df_postal.get('최초 접수월', ym[1]).fillna(ym[1])
for c in ['피해액', '최초 접수년', '최초 접수월', '연령대']:
    if c in df_postal:
        df_postal[c] = pd.to_numeric(df_postal[c].astype('string').str.replace(',', '', regex=False), errors='coerce')
if '연령대' in df_postal:
    df_postal.loc[df_postal['연령대'].eq(0), '연령대'] = np.nan

# 원본파일/기준일을 제외한 값이 같은 행을 중복 후보로 표시하되 삭제하지 않음
compare_cols = [c for c in df_postal.columns if c not in {'원본파일', '자료기준일'}]
df_postal['중복후보'] = df_postal.duplicated(compare_cols, keep=False)
# 최종 데이터는 모두 전화 사례이며, 이 플래그는 신청사유가 보이스피싱인지 표시
df_postal['전화_보이스피싱'] = (
    df_postal.get('접근매체', pd.Series(index=df_postal.index, dtype='string')).eq('전화') &
    df_postal.get('피해구제 신청사유', pd.Series(index=df_postal.index, dtype='string')).eq('보이스피싱')
)
display(filter_summary)
print('최종 피해구제 신청사유 분포:')
display(df_postal['피해구제 신청사유'].value_counts(dropna=False).to_frame('건수'))
print('df_postal:', df_postal.shape)
display(df_postal.head())

## 2. 경찰청 집계 자료 통합

서로 다른 집계 단위를 한 행으로 조인하지 않고 `연도·지역·분류·지표·값·단위`의 긴 형식으로 쌓습니다.

In [ ]:
police_parts = []

def add_police(rows, source):
    z = rows.copy()
    z['출처파일'] = source
    police_parts.append(z)

for name, df in tables.items():
    cols = set(df.columns)
    if {'기관사칭형_발생건수', '대출사기형_발생건수'}.issubset(cols):
        for fraud_type in ['기관사칭형', '대출사기형']:
            for metric, suffix, unit in [('발생건수', '발생건수', '건'), ('피해액', '피해액_억원', '억원'), ('검거인원', '검거인원', '명')]:
                c = f'{fraud_type}_{suffix}'
                z = pd.DataFrame({'연도': df['구분'], '지역': '전국', '분류축': '사기유형',
                                  '분류값': fraud_type, '지표': metric, '값': df[c], '단위': unit})
                add_police(z, name)
    elif {'남성', '여성'}.issubset(cols):
        z = df.melt(id_vars='구분', value_vars=['남성', '여성'], var_name='분류값', value_name='값')
        z = z.rename(columns={'구분': '연도'}).assign(지역='전국', 분류축='성별', 지표='피해자수', 단위='명')
        add_police(z[['연도','지역','분류축','분류값','지표','값','단위']], name)
    elif {'20대이하', '30대', '40대', '50대', '60대', '70대이상'}.issubset(cols):
        age_cols = ['20대이하', '30대', '40대', '50대', '60대', '70대이상']
        z = df.melt(id_vars='구분', value_vars=age_cols, var_name='분류값', value_name='값')
        z = z.rename(columns={'구분': '연도'}).assign(지역='전국', 분류축='연령대', 지표='피해자수', 단위='명')
        add_police(z[['연도','지역','분류축','분류값','지표','값','단위']], name)
    elif {'년', '월', '전화금융사기 발생건수'}.issubset(cols):
        z = pd.DataFrame({'연도': df['년'], '월': df['월'], '지역': '전국', '분류축': '전체',
                          '분류값': '전체', '지표': '발생건수', '값': df['전화금융사기 발생건수'], '단위': '건'})
        add_police(z, name)
    elif '시도청' in cols and any(re.fullmatch(r'20\d{2}년', c) for c in df.columns):
        year_cols = [c for c in df.columns if re.fullmatch(r'20\d{2}년', c)]
        z = df.melt(id_vars='시도청', value_vars=year_cols, var_name='연도', value_name='값')
        z['연도'] = z['연도'].str.extract(r'(20\d{2})')[0]
        z = z.rename(columns={'시도청': '지역'}).assign(분류축='전체', 분류값='전체', 지표='피해액', 단위='억원')
        add_police(z[['연도','지역','분류축','분류값','지표','값','단위']], name)

assert police_parts, '경찰청 집계 형식의 CSV를 찾지 못했습니다.'
df_police = pd.concat(police_parts, ignore_index=True)
df_police['연도'] = pd.to_numeric(df_police['연도'], errors='coerce').astype('Int64')
df_police['월'] = pd.to_numeric(df_police.get('월'), errors='coerce').astype('Int64')
df_police['값'] = pd.to_numeric(df_police['값'].astype('string').str.replace(',', '', regex=False), errors='coerce')
df_police = df_police.sort_values(['연도','월','지역','분류축','분류값','지표'], na_position='last').reset_index(drop=True)
print('df_police:', df_police.shape)
display(df_police.head(12))

## 3. 품질검증

In [ ]:
# 연령대/성별 피해자 합계와 유형별 발생건수 합계를 비교
age_total = df_police.query("분류축 == '연령대' and 지표 == '피해자수'").groupby('연도')['값'].sum()
gender_total = df_police.query("분류축 == '성별' and 지표 == '피해자수'").groupby('연도')['값'].sum()
case_total = df_police.query("분류축 == '사기유형' and 지표 == '발생건수'").groupby('연도')['값'].sum()
qc_annual = pd.concat([case_total.rename('발생건수'), age_total.rename('연령합계'), gender_total.rename('성별합계')], axis=1)
qc_annual['연령차이'] = qc_annual['연령합계'] - qc_annual['발생건수']
qc_annual['성별차이'] = qc_annual['성별합계'] - qc_annual['발생건수']

regional = df_police.query("지역 != '전국' and 지표 == '피해액'").groupby('연도')['값'].sum()
national = df_police.query("지역 == '전국' and 지표 == '피해액'").groupby('연도')['값'].sum()
qc_damage = pd.concat([regional.rename('시도청합계'), national.rename('전국유형합계')], axis=1).dropna()
qc_damage['차이_억원'] = qc_damage['시도청합계'] - qc_damage['전국유형합계']

display(qc_annual)
display(qc_damage)
print('우체국 결측치 상위:')
display(df_postal.isna().sum().sort_values(ascending=False).head(10).to_frame('결측수'))

## 4. 두 데이터프레임과 검증표 저장

In [ ]:
out_dir = Path('voice_phishing_outputs_filtered')
out_dir.mkdir(exist_ok=True)
df_postal.to_csv(out_dir / 'df_postal.csv', index=False, encoding='utf-8-sig')
df_police.to_csv(out_dir / 'df_police.csv', index=False, encoding='utf-8-sig')
df_manifest.to_csv(out_dir / 'file_manifest.csv', index=False, encoding='utf-8-sig')
qc_annual.to_csv(out_dir / 'qc_annual_totals.csv', encoding='utf-8-sig')
qc_damage.to_csv(out_dir / 'qc_damage_totals.csv', encoding='utf-8-sig')
filter_summary.to_csv(out_dir / 'qc_postal_filter.csv', index=False, encoding='utf-8-sig')

zip_path = Path('voice_phishing_integrated_data_filtered.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in out_dir.glob('*.csv'):
        zf.write(p, arcname=p.name)
print('저장 완료:', zip_path.resolve())
try:
    files.download(str(zip_path))
except NameError:
    pass